# Extension-method tree, ported to Python (pandas + PyROOT `Hypfit`)

This notebook reproduces `make_extensionMethodTree_min` from the ROOT macro, but reads the
per-hit / per-particle information from the `cafpyana` dataframes instead of the intermediate
`tree.root` file, and reuses the already-compiled `Hypfit`/`PhysdEdx` C++ classes through PyROOT
(the loader snippet you already have, producing the `h_fit` object).

**Assumptions you should double-check** (I don't have your `mc_bnb_pfp_df` columns, only the
hit dataframes):

- `mc_bnb_pfp_df` has one row per particle, indexed by
  `['__ntuple', 'entry', 'rec.slc..index', 'rec.slc.reco.pfp..index']`, and contains columns
  equivalent to the C++ tree branches: `true_P`, `range_P`, `length`, `purity`, `completeness`,
  `PDG`, `end_process_string`, `best_plane`. If your real column names differ (e.g. they might be
  MultiIndex columns like `('truth','p',...)` in cafpyana), just edit the `COLS` dict in the
  config cell below — everything else references that dict, not the literal names.
- `hit0_df`/`hit1_df`/`hit2_df` correspond to plane 0/1/2, and share the first four index levels
  with `pfp_df`; the fifth level is the hit index (hit0, hit1, ... within that track/plane).
- `rr` in the hit dataframes is residual range (same role as `rr_vec` in the macro).
- `h_fit` (a `ROOT.Hypfit()` instance) is already instantiated in this kernel, exactly like your
  loader snippet does — this notebook does **not** redo that loading, it just uses `h_fit`.

In [ ]:
import os, glob

base_path = "."  # match your existing base_path

# Remove every ACLiC-generated artifact for both classes
patterns = [
    "PhysdEdx_cpp*", "Hypfit_cpp*",
    "PhysdEdx_ACLiC_dict*", "Hypfit_ACLiC_dict*",
    "*.pcm", "*.d",
]
for pat in patterns:
    for f in glob.glob(os.path.join(base_path, pat)):
        print("removing", f)
        os.remove(f)

In [ ]:
# Absolute path to cafpyana directory
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import os
import ROOT
import subprocess
from ROOT import std

# 1. Setup Paths
base_path = "/home/lpelegri/cafpyana/analysis_village/cc1pi/TLExtensionMethod"
# Let's get the ROOT lib directory dynamically to ensure it's correct
try:
    root_lib_dir = subprocess.check_output(['root-config', '--libdir'], text=True).strip()
except:
    root_lib_dir = "/cvmfs/larsoft.opensciencegrid.org/products/root/v6_28_12/Linux64bit+3.10-2.17-e26-p3915-prof/lib"

# 2. Update Environment
os.environ['LD_LIBRARY_PATH'] = f"{root_lib_dir}:{os.environ.get('LD_LIBRARY_PATH', '')}"
ROOT.gSystem.AddDynamicPath(root_lib_dir)

# 3. Load the "Big" ROOT dependency blocks
# Loading these covers almost all physics class requirements (Hist, Geom, Graf, etc.)
root_libs = ["libRint","libROOTGpadv7","libMathMore","libCore", "libRIO", "libNet", "libHist", "libGraf", "libGraf3d", "libGpad", "libTree", "libMathCore", "libThread", "libMatrix", "libGeom","libROOTHist"]
for lib in root_libs:
    ROOT.gSystem.Load(lib)

def load_custom_class(class_name):
    source_file = os.path.join(base_path, f"{class_name}.cpp")
    header_file = os.path.join(base_path, f"{class_name}.h")
    so_file = os.path.join(base_path, f"{class_name}_cpp.so")

    status = ROOT.gSystem.CompileMacro(source_file, "kOf")
    if status >= 0:
        if ROOT.gSystem.Load(so_file) < 0:
            print(f"⚠️  Compiled {class_name} but failed to load {so_file}")
            return False
        ROOT.gInterpreter.Declare(f'#include "{header_file}"')  # keep this!
        print(f"📦 Successfully compiled and linked: {class_name}")
        return True
    else:
        print(f"❌ Failed to compile: {class_name}")
        return False

if load_custom_class("PhysdEdx"):
    if load_custom_class("Hypfit"):
        try:
            ROOT.gInterpreter.ProcessLine(f'#include "{os.path.join(base_path, "Hypfit.h")}"')
            h_fit = ROOT.Hypfit()
            print("🚀 Success! Hypfit object initialized and linked.")
        except Exception as e:
            print(f"❌ Error during instantiation: {e}")
            ROOT.gInterpreter.ProcessLine("Hypfit* h_fit_ptr = new Hypfit();")
            h_fit = ROOT.h_fit_ptr
            print("🚀 Success! Hypfit object initialized via Global Pointer fallback.")

In [ ]:
import numpy as np
import pandas as pd
import ROOT
from ROOT import std
import uproot
import awkward as ak

# h_fit must already exist in the kernel (from your Hypfit loader cell). Sanity check:
assert 'h_fit' in globals(), "Run your Hypfit/PhysdEdx loader cell first so `h_fit` exists."

In [ ]:

from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

keys2load = ["pfp", "hdr", "histpotdf","hit0","hit1","hit2"] ## keys from the configuration file
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/"
#file = "TLE_1e20_pion_all_update_calo.df"
file = "TLE_5e18_muon_data_update_calo.df"

mc_bnb_df = load_df(bnb_path + file, keys2load, 20)
mc_bnb_pfp_df = mc_bnb_df['pfp']
mc_bnb_hit0_df = mc_bnb_df['hit0']
mc_bnb_hit1_df = mc_bnb_df['hit1']
mc_bnb_hit2_df = mc_bnb_df['hit2']
mc_bnb_hdr_df = mc_bnb_df['hdr']

In [ ]:
muon_mass   = 0.106583755
pion_mass   = 0.13957039
proton_mass = 0.9382720881

PDG = 13
if "pion" in file:
    PDG = 211        
use_data = "data" in file

check_pur_comp   = False
make_energy_cut  = False

selected_plane = 2
use_best_plane = True

subtract_hits = True
n_skip = 25
if PDG == 211:
    n_skip = 10
    
# Column names to pull off mc_bnb_pfp_df -- EDIT THESE to match your real pfp_df columns.
COLS = {
    "true_P":       "true_P",
    "range_P":      "range_P",
    "length":       "length",
    "purity":       "purity",
    "completeness": "completeness",
    "PDG":          "PDG",
    "end_process":  "end_process_string",
    "best_plane":   "best_plane",
}

# Likelihood_preset struct -> list of dicts (same entries the macro has active/uncommented)
presets = [
    {"preset_key": "range", "method": "range",
     "cleaning_method": "skip_only", "normalize": False, "include_small_likelihood": False},
    {"preset_key": "original", "method": "likelihood",
     "cleaning_method": "skip_only_truncate_harsh", "normalize": False, "include_small_likelihood": False},
    {"preset_key": "normalize_clean_dEdx_include_small", "method": "likelihood",
     "cleaning_method": "all", "normalize": True, "include_small_likelihood": True},
    {"preset_key": "convolution_normalize_clean_dEdx_include_small_og", "method": "convolution_og",
     "cleaning_method": "all", "normalize": True, "include_small_likelihood": True},
    {"preset_key": "convolution_normalize_clean_dEdx_include_small_shifted", "method": "convolution_shifted",
     "cleaning_method": "all", "normalize": True, "include_small_likelihood": True},
    {"preset_key": "langau_func_normalize_clean_dEdx_include_small", "method": "langau",
     "cleaning_method": "all", "normalize": True, "include_small_likelihood": True},
]


mass_by_pdg = {13: muon_mass, 211: pion_mass, 2212: proton_mass}
'''

    {"preset_key": "convolution_normalize_clean_dEdx_include_small_w_x_og",
     "cleaning_method": "all",
     "normalize": True, "include_small_likelihood": True},
    
    {"preset_key": "convolution_normalize_clean_dEdx_include_small_w_x_shifted",
     "cleaning_method": "all",
     "normalize": True, "include_small_likelihood": True},

    

     
    {"preset_key": "convolution_normalize_clean_dEdx_include_small_og_w_pitch",
     "cleaning_method": "all",
     "normalize": True, "include_small_likelihood": True},

    {"preset_key": "convolution_normalize_clean_dEdx_include_small_shifted_w_pitch",
     "cleaning_method": "all",
     "normalize": True, "include_small_likelihood": True},
{"preset_key": "convolution_normalize_clean_dEdx_include_small_shifted_each_time",
 "cleaning_method": "all",
 "normalize": True, "include_small_likelihood": True},


{"preset_key": "convolution_normalize_clean_dEdx_include_small_og_each_time",
 "cleaning_method": "all",
 "normalize": True, "include_small_likelihood": True},
'''

In [ ]:
conv_tf1_map                    = h_fit.get_conv_function_map(PDG, "og", 500)
conv_shifted_tf1_map            = h_fit.get_conv_function_map(PDG, "shift", 500)

conv_tf1_map_data                    = h_fit.get_conv_function_map(PDG, "og", 500, True)
conv_shifted_tf1_map_data            = h_fit.get_conv_function_map(PDG, "shift", 500, True)

''''
# New: w_x_ranges versions
conv_tf1_map_w_x_ranges         = h_fit.get_conv_function_map_w_x_ranges(PDG, "og", 500)
conv_shifted_tf1_map_w_x_ranges = h_fit.get_conv_function_map_w_x_ranges(PDG, "shift", 500)

conv_tf1_map_w_pitch                        = h_fit.get_conv_function_map_w_pitch(PDG, "none", 750)
conv_shifted_tf1_map_w_pitch                 = h_fit.get_conv_function_map_w_pitch(PDG, "shift", 750)
'''

In [ ]:
def to_std_vector_double(arr):
    return ROOT.std.vector('double')(np.ascontiguousarray(arr, dtype=np.float64))


def get_track_hits(particle_idx, plane, hit_dfs):
    """Return (rr, dedx, pitch, x) numpy arrays for one particle on one plane, sorted by rr,
    with non-finite entries dropped -- the pandas equivalent of rr_vec->at(plane) etc.
    Returns None if this particle has no hits on that plane."""
    hit_df = hit_dfs[plane]
    try:
        hits = hit_df.loc[particle_idx]
    except KeyError:
        return None
    if isinstance(hits, pd.Series):
        # a single hit row -- promote to a 1-row frame
        hits = hits.to_frame().T
    hits = hits.sort_values('rr')
    rr    = hits['rr'].to_numpy(dtype=np.float64)
    dedx  = hits['dedx'].to_numpy(dtype=np.float64)
    pitch = hits['pitch'].to_numpy(dtype=np.float64)
    x     = hits['x'].to_numpy(dtype=np.float64)
    mask = np.isfinite(rr) & np.isfinite(dedx) & np.isfinite(pitch) & np.isfinite(x)
    return rr[mask], dedx[mask], pitch[mask], x[mask]


def to_std_vector_of_tf1_vec(list_of_tf1_vecs):
    """Convert a Python list of vector<TF1*> (one per x-range) into a
    std::vector<std::vector<TF1*>> for NormLikelihood_w_convolution_w_x_ranges."""
    outer = std.vector('vector<TF1*>')()
    for tf1_vec in list_of_tf1_vecs:
        outer.push_back(tf1_vec)
    return outer
    
def compute_reco_KE_P(preset, rr, dedx, pitch, x, length, used_plane, PDG, use_data):
    """Direct port of the big if/else block inside the preset loop of the macro."""
    v_rr    = to_std_vector_double(rr)
    v_dedx  = to_std_vector_double(dedx)
    v_pitch = to_std_vector_double(pitch)
    v_x     = to_std_vector_double(x)
    bad_hits = h_fit.get_hits_to_ignore(v_rr, v_dedx, preset["cleaning_method"])
    method = preset["method"]

    # Pick data- or MC-derived maps once, up front.
    this_conv_tf1_map         = conv_tf1_map_data if use_data else conv_tf1_map
    this_conv_shifted_tf1_map = conv_shifted_tf1_map_data if use_data else conv_shifted_tf1_map
    this_langau_tf1_map       = langau_map_data if use_data else langau_tf1_map

    if method == "range":
        if len(rr) - 1 == 0:
            reco_KE = h_fit.map_PhysdEdx[PDG].KEFromRangeSpline(length)
        else:
            reco_KE = h_fit.map_PhysdEdx[PDG].KEFromRangeSpline(float(rr[-1]))

    elif method == "convolution_og":
        reco_KE = h_fit.NormLikelihood_w_convolution(
            v_dedx, v_rr, v_pitch, bad_hits, True, PDG, this_conv_tf1_map[used_plane])

    elif method == "convolution_shifted":
        reco_KE = h_fit.NormLikelihood_w_convolution(
            v_dedx, v_rr, v_pitch, bad_hits, True, PDG, this_conv_shifted_tf1_map[used_plane])

    elif method == "langau":
        reco_KE = h_fit.NormLikelihood_w_convolution(
            v_dedx, v_rr, v_pitch, bad_hits, True, PDG, this_langau_tf1_map[used_plane])

    elif method == "likelihood":
        if preset["normalize"]:
            reco_KE = h_fit.NormLikelihood(
                v_dedx, v_rr, v_pitch, bad_hits, preset["include_small_likelihood"], PDG)
        else:
            reco_KE = h_fit.Likelihood(v_dedx, v_rr, v_pitch, bad_hits, PDG)

    else:
        raise ValueError(f"Unknown preset method: {method!r} (preset_key={preset['preset_key']!r})")

    reco_P = h_fit.map_PhysdEdx[PDG].KEtoMomentum(reco_KE) / 1000.0
    v_rr.clear(); v_dedx.clear(); v_pitch.clear(); v_x.clear()
    del v_rr, v_dedx, v_pitch, v_x
    return reco_KE, reco_P

In [ ]:
import numpy as np
import pandas as pd

PROCESS_MAP = {
    0: "primary",
    1: "CoupledTransportation",
    2: "FastScintillation",
    3: "Decay",
    4: "anti_neutronInelastic",
    5: "neutronInelastic",
    6: "anti_protonInelastic",
    7: "protonInelastic",
    8: "hadInelastic",
    9: "pipInelastic",
    10: "pimInelastic",
    11: "xipInelastic",
    12: "ximInelastic",
    13: "kaonpInelastic",
    14: "kaonmInelastic",
    15: "sigmapInelastic",
    16: "sigmamInelastic",
    17: "kaon0LInelastic",
    18: "kaon0SInelastic",
    19: "lambdaInelastic",
    20: "anti_lambdaInelastic",
    21: "He3Inelastic",
    22: "ionInelastic",
    23: "xi0Inelastic",
    24: "alphaInelastic",
    25: "tInelastic",
    26: "dInelastic",
    27: "anti_neutronElastic",
    28: "neutronElastic",
    29: "anti_protonElastic",
    30: "protonElastic",
    31: "hadElastic",
    32: "pipElastic",
    33: "pimElastic",
    34: "kaonpElastic",
    35: "kaonmElastic",
    36: "conv",
    37: "phot",
    38: "annihil",
    39: "nCapture",
    40: "nKiller",
    41: "muMinusCaptureAtRest",
    42: "muIoni",
    43: "eBrem",
    44: "CoulombScat",
    45: "hBertiniCaptureAtRest",
    46: "hFritiofCaptureAtRest",
    47: "photonNuclear",
    48: "muonNuclear",
    49: "electronNuclear",
    50: "positronNuclear",
    51: "compt",
    52: "eIoni",
    53: "muBrems",
    54: "hIoni",
    55: "muPairProd",
    56: "hPairProd",
    57: "LArVoxelReadoutScoringProcess",
    58: "ionIoni",
    59: "hBrems",
    60: "Transportation",
    61: "msc",
    62: "StepLimiter",
    63: "LegacyUNKNOWN",
    64: "RadioactiveDecayBase",
    1024: "UNKNOWN",
}

def get_process_name(code):
    if code is None or pd.isna(code):
        return "UNKNOWN"
    try:
        return PROCESS_MAP.get(int(code), "UNKNOWN")
    except (ValueError, TypeError):
        return "UNKNOWN"

def process_particle(
    particle_idx,
    pfp_row,
    hits,
    best_plane,
    used_plane,
    presets,
    PDG,
    make_energy_cut,
    subtract_hits,
    n_skip,
):
    rows = []
    rr, dedx, pitch, x = hits   # <-- now unpacks 4
    
    num_initial_hits = len(pitch)
    if num_initial_hits == 1:
        num_initial_hits = 0
        
    length = pfp_row.get(COLS["length"], np.nan)
    range_P = pfp_row.get(COLS["range_P"], np.nan)
    
    raw_end_process = pfp_row.get("end_process", None)
    end_process_string = get_process_name(raw_end_process)
    
    base_info = dict(
        particle_idx=particle_idx,
        true_P=pfp_row.get(COLS["true_P"], np.nan),
        range_P=range_P,
        length=length,
        purity=pfp_row.get(COLS["purity"], np.nan),
        completeness=pfp_row.get(COLS["completeness"], np.nan),
        pdg=pfp_row.get(COLS["PDG"], PDG),
        end_process_string=end_process_string,
        best_plane=best_plane,
        used_plane=used_plane,
        num_initial_hits=num_initial_hits,
    )
    if make_energy_cut and pd.notna(range_P) and range_P > 0.8:
        row = dict(base_info)
        row.update(
            iteration=0,
            num_used_hits=0,
            num_removed_hits=0,
            method_vector=[p["preset_key"] for p in presets],
            P_track_extension_method=[121212.0] * len(presets),
            exit_code=[-121212] * len(presets),
            rr_vec=rr.tolist(),
            dEdx_vec=dedx.tolist(),
            pitch_vec=pitch.tolist(),
        )
        rows.append(row)
        return rows
    this_rr, this_dedx, this_pitch, this_x = rr.copy(), dedx.copy(), pitch.copy(), x.copy()
    exit_loop = False
    n_it = 0
    MAX_ITERATIONS = 1000
    
    while not exit_loop and n_it < MAX_ITERATIONS:
        n_it += 1
        num_used_hits = len(this_rr)
        num_removed_hits = num_initial_hits - num_used_hits

        method_vector, P_vals, exit_codes = [], [], []
        for preset in presets:
            reco_KE, reco_P = compute_reco_KE_P(
                preset,
                this_rr,
                this_dedx,
                this_pitch,
                this_x,
                length,
                used_plane,
                PDG,
                use_data,   # <-- new
            )
            method_vector.append(preset["preset_key"])
            P_vals.append(reco_P)
            exit_codes.append(0 if reco_KE >= 0 else int(reco_KE))

        row = dict(base_info)
        row.update(
            iteration=n_it,
            num_used_hits=num_used_hits,
            num_removed_hits=num_removed_hits,
            method_vector=method_vector,
            P_track_extension_method=P_vals,
            exit_code=exit_codes,
            rr_vec=this_rr.tolist(),
            dEdx_vec=this_dedx.tolist(),
            pitch_vec=this_pitch.tolist(),
            x_vec=this_x.tolist(),   # optional, but worth keeping for debugging/plots
        )
        rows.append(row)

        if not subtract_hits or len(this_rr) <= n_skip:
            exit_loop = True
        else:
            this_rr = this_rr[n_skip:]
            this_dedx = this_dedx[n_skip:]
            this_pitch = this_pitch[n_skip:]
            this_x = this_x[n_skip:]
            if len(this_rr) == 0:
                exit_loop = True
            else:
                this_rr = this_rr - this_rr[0]

    return rows

In [ ]:
print(mc_bnb_pfp_df.columns)

In [ ]:
rename_map = {
    ("pfp", "trk", "len", "", "", ""): "length",
    ("pfp", "trk", "rangeP", "p_muon", "", ""): "range_P",  # Change to 'p_pion' if analyzing pions
    ("pfp", "trk", "truth", "genp", "mag", ""): "true_P",
    ("pfp", "trk", "truth", "pur", "", ""): "purity",
    ("pfp", "trk", "truth", "eff", "", ""): "completeness",
    ("pfp", "trk", "truth", "p", "pdg", ""): "PDG",
    ("pfp", "trk", "truth", "p", "end_process", ""): "end_process",
    ("pfp", "best_plane", "", "", "", ""): "best_plane",
}
if "pion" in bnb_path:
    rename_map = {
        ("pfp", "trk", "len", "", "", ""): "length",
        ("pfp", "trk", "rangeP", "p_pion", "", ""): "range_P",  # Change to 'p_pion' if analyzing pions
        ("pfp", "trk", "truth", "genp", "mag", ""): "true_P",
        ("pfp", "trk", "truth", "pur", "", ""): "purity",
        ("pfp", "trk", "truth", "eff", "", ""): "completeness",
        ("pfp", "trk", "truth", "p", "pdg", ""): "PDG",
        ("pfp", "trk", "truth", "p", "end_process", ""): "end_process",
        ("pfp", "best_plane", "", "", "", ""): "best_plane",
    }

# Direct column reassignment bypasses MultiIndex structural restrictions
mc_bnb_pfp_df.columns = [
    rename_map.get(col, col) for col in mc_bnb_pfp_df.columns
]

# List of renamed columns to keep
target_columns = [
    "length",
    "range_P",
    "true_P",
    "purity",
    "completeness",
    "PDG",
    "end_process",
    "best_plane",
]

# Filter the DataFrame to keep only these target columns
mc_bnb_pfp_df = mc_bnb_pfp_df[
    [c for c in target_columns if c in mc_bnb_pfp_df.columns]
]

In [ ]:
len(mc_bnb_pfp_df[mc_bnb_pfp_df.end_process == 9])

In [ ]:
mc_bnb_pfp_df.columns

In [ ]:
import os
import time
import warnings
import threading
from concurrent.futures import ProcessPoolExecutor
import numpy as np
import pandas as pd
import psutil
from pandas.errors import PerformanceWarning
from tqdm import tqdm
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import gc


warnings.filterwarnings("ignore", category=PerformanceWarning)
n_cores =  max(1, int(os.cpu_count() * 0.8))

df_filtered = mc_bnb_pfp_df
hit_dfs = {
    0: mc_bnb_hit0_df,
    1: mc_bnb_hit1_df,
    2: mc_bnb_hit2_df,
}
print("Sorting hit DataFrame indices...")
hit_dfs = {k: v.sort_index() for k, v in hit_dfs.items()}

if check_pur_comp:
    pur_col = COLS["purity"]
    comp_col = COLS["completeness"]
    pur_mask = df_filtered[pur_col].isna() | (df_filtered[pur_col] >= 0.8)
    comp_mask = df_filtered[comp_col].isna() | (df_filtered[comp_col] >= 0.8)
    df_filtered = df_filtered[pur_mask & comp_mask]

# Pass ONLY particle indices across the queue to avoid serialization IPC bloat
tasks = list(df_filtered.index)



gc.collect()
gc.freeze()

def _worker_init():
    # Per-task objects here (numpy arrays, small dict rows) are short-lived and
    # essentially acyclic, so there's nothing worth collecting in the child --
    # and this guarantees no further COW-defeating refcount walks happen at all.
    gc.disable()

def _worker(particle_idx):
    t0 = time.time()
    
    # Read directly from fork-inherited DataFrame using particle_idx
    pfp_dict = df_filtered.loc[particle_idx]
    
    best_plane = pfp_dict.get(COLS["best_plane"], -1)
    best_plane = int(best_plane) if pd.notna(best_plane) else -1

    if use_best_plane:
        if best_plane < 0 or best_plane >= 3:
            return [], time.time() - t0, psutil.Process().memory_info().rss
        used_plane = best_plane
    else:
        if selected_plane < 0 or selected_plane >= 3:
            return [], time.time() - t0, psutil.Process().memory_info().rss
        used_plane = selected_plane

    hits = get_track_hits(particle_idx, used_plane, hit_dfs)
    if hits is None:
        return [], time.time() - t0, psutil.Process().memory_info().rss

    result = process_particle(
        particle_idx,
        pfp_dict,
        hits,
        best_plane,
        used_plane,
        presets,
        PDG,
        make_energy_cut,
        subtract_hits,
        n_skip,
    )
    return result, time.time() - t0, psutil.Process().memory_info().rss


all_rows = []
dts = []
mems = []
lock = threading.Lock()

print(f"Processing {len(tasks)} tasks on {n_cores} CPU cores...")
with ProcessPoolExecutor(max_workers=n_cores, initializer=_worker_init) as executor:
    tasks_iter = iter(tasks)
    pbar = tqdm(total=len(tasks), desc="Processing particles", mininterval=0.5, file=sys.stdout)

    def _on_done(fut):
        res, dt, rss = fut.result()
        with lock:
            dts.append(dt)
            mems.append(rss)
            if res:
                all_rows.extend(res)
            pbar.update(1)
            try:
                new_fut = executor.submit(_worker, next(tasks_iter))
                new_fut.add_done_callback(_on_done)
            except StopIteration:
                pass
    for _ in range(n_cores):
        try:
            fut = executor.submit(_worker, next(tasks_iter))
            fut.add_done_callback(_on_done)
        except StopIteration:
            break
    while len(dts) < len(tasks):
        time.sleep(0.05)
    pbar.close()
    

output_df = pd.DataFrame(all_rows)
print(f"Built {len(output_df)} output rows from {len(mc_bnb_pfp_df)} particles.")

output_df = pd.DataFrame(all_rows)
print(f"Built {len(output_df)} output rows from {len(mc_bnb_pfp_df)} particles.")

# --- per-task timing stats ---
dts = np.array(dts)
mems = np.array(mems) / 1e9  # GB

print(f"Task time stats (s): mean={dts.mean():.4f}  median={np.median(dts):.4f}  "
      f"p90={np.percentile(dts, 90):.4f}  p99={np.percentile(dts, 99):.4f}  max={dts.max():.4f}")
print(f"Worker RSS stats (GB): mean={mems.mean():.3f}  median={np.median(mems):.3f}  "
      f"p90={np.percentile(mems, 90):.3f}  max={mems.max():.3f}")

# --- plots ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(dts, bins=50)
axes[0].set_xlabel("Task duration (s)")
axes[0].set_ylabel("Count")
axes[0].set_title("Per-task duration distribution")

axes[1].plot(dts, marker=".", linestyle="none", alpha=0.5)
axes[1].set_xlabel("Task completion order")
axes[1].set_ylabel("Duration (s)")
axes[1].set_title("Task duration over completion order")

axes[2].scatter(range(len(mems)), mems, s=8, alpha=0.5)
axes[2].set_xlabel("Task completion order")
axes[2].set_ylabel("Worker RSS (GB)")
axes[2].set_title("Worker memory over completion order")

plt.tight_layout()
plt.show()


In [ ]:
import array
import ROOT
'''
_selection_result_0_32_pdf_fixed_shift_double_fit
'''
output_file = f"/exp/sbnd/data/users/lpelegri/TLE_processed_files/{PDG}_{'data' if use_data else 'MC'}"
output_file += "_subtracted" if subtract_hits else ""
output_file += " _area_fit.root"

fout = ROOT.TFile(output_file, "RECREATE")
tree = ROOT.TTree("tree", "tree")

# 1. Allocate C++ scalar buffers
out_true_P = array.array("d", [0.0])
out_range_P = array.array("d", [0.0])
out_length = array.array("d", [0.0])
out_purity = array.array("d", [0.0])
out_completeness = array.array("d", [0.0])

out_pdg = array.array("i", [0])
out_best_plane = array.array("i", [0])
out_used_plane = array.array("i", [0])
num_initial_hits = array.array("i", [0])
num_used_hits = array.array("i", [0])
num_removed_hits = array.array("i", [0])

# 2. Allocate C++ object/vector buffers
out_end_process_string = ROOT.std.string()
method_vector = ROOT.std.vector("std::string")()
P_track_extension_method = ROOT.std.vector("double")()
extended_rr = ROOT.std.vector("double")()
exit_code = ROOT.std.vector("int")()

# 3. Create Tree Branches matching your C++ types
tree.Branch("true_P", out_true_P, "true_P/D")
tree.Branch("range_P", out_range_P, "range_P/D")
tree.Branch("length", out_length, "length/D")
tree.Branch("purity", out_purity, "purity/D")
tree.Branch("completeness", out_completeness, "completeness/D")

tree.Branch("PDG", out_pdg, "PDG/I")
tree.Branch("end_process_string", out_end_process_string)
tree.Branch("out_best_plane", out_best_plane, "out_best_plane/I")
tree.Branch("used_plane", out_used_plane, "used_plane/I")
tree.Branch("num_used_hits", num_used_hits, "num_used_hits/I")
tree.Branch("num_initial_hits", num_initial_hits, "num_initial_hits/I")
tree.Branch("num_removed_hits", num_removed_hits, "num_removed_hits/I")

tree.Branch("method_vector", method_vector)
tree.Branch("P_track_extension_method", P_track_extension_method)
tree.Branch("extended_rr", extended_rr)
tree.Branch("exit_code", exit_code)

# 4. Fill the TTree
for _, row in output_df.iterrows():
    out_true_P[0] = float(row.get("true_P", 0.0))
    out_range_P[0] = float(row.get("range_P", 0.0))
    out_length[0] = float(row.get("length", 0.0))
    out_purity[0] = float(row.get("purity", 0.0))
    out_completeness[0] = float(row.get("completeness", 0.0))

    out_pdg[0] = int(row.get("pdg", 0))
    out_best_plane[0] = int(row.get("best_plane", -1))
    out_used_plane[0] = int(row.get("used_plane", -1))
    num_initial_hits[0] = int(row.get("num_initial_hits", 0))
    num_used_hits[0] = int(row.get("num_used_hits", 0))
    num_removed_hits[0] = int(row.get("num_removed_hits", 0))

    out_end_process_string.assign(str(row.get("end_process_string", "")))

    # Clear and fill std::vector<std::string>
    method_vector.clear()
    for item in row.get("method_vector", []):
        method_vector.push_back(str(item))

    # Clear and fill std::vector<double>
    P_track_extension_method.clear()
    for item in row.get("P_track_extension_method", []):
        P_track_extension_method.push_back(float(item))

    extended_rr.clear()
    for item in row.get("rr_vec", []):
        extended_rr.push_back(float(item))

    # Clear and fill std::vector<int>
    exit_code.clear()
    for item in row.get("exit_code", []):
        exit_code.push_back(int(item))

    tree.Fill()

fout.Write()
fout.Close()
print(f"Wrote {output_file}")